# Design of experiments

A design of experiments (DOE) evaluates a model at a set of points rather than solving it. Each
**factor** $x$ is set to every point of its **plan**, $x_i = p(i)$ for $i = 1, \dots, n$, and
each **response** $h$ is recorded there, $h_i = F(x_i)$, $F$ being everything the DOE's cycle
computes. Here we sweep the Sellar analysis,

$$
\begin{aligned}
y_1 &= z_1^2 + z_2 + x_1 - 0.2\, y_2\\
y_2 &= \sqrt{y_1} + z_1 + z_2\\
f   &= x_1^2 + z_2 + y_1 + \exp(-y_2)
\end{aligned}
$$

over a full factorial grid in $z_1$ and $z_2$, recording $f$, $y_1$ and $y_2$ at every point. The
coupling between $y_1$ and $y_2$ is solved at each point.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import cottax.assemblies.jax as cx
from cottax.assemblies.jax import Start
flat = cx.flat

In [ ]:
@flat.function('y_1')
def D1(x_1, z_1, z_2, y_2):
    return z_1**2 + z_2 + x_1 - 0.2 * y_2

@flat.function('y_2')
def D2(y_1, z_1, z_2):
    return jnp.sqrt(y_1) + z_1 + z_2

@flat.function('f')
def obj(x_1, z_2, y_1, y_2):
    return x_1**2 + z_2 + y_1 + jnp.exp(-y_2)

## The plan

The plan $p$ is an ordinary model upstream of the DOE: it writes, per factor, an array of the
points along its leading axis. Here a full factorial, `N` levels per factor in $[0, 4]$. `n`
sizes arrays, so it is given as `cx.Static`: kept in the jit key rather than traced.

In [ ]:
N = 9

@flat.function('z_1s', 'z_2s', reads={'n': cx.Static(N)})
def grid(n):
    levels = jnp.linspace(0., 4., n)
    z_1s, z_2s = jnp.meshgrid(levels, levels, indexing='ij')
    return z_1s.ravel(), z_2s.ravel()

## The DOE

`flat.sample` declares the DOE, in a slot (here `sweep`).

`over` maps each factor to where its plan is read: `z_1` takes its points from `z_1s` and `z_2`
from `z_2s`, the two arrays `grid` writes. An array written there instead,
`over={'z_1': jnp.linspace(0., 4., N)}`, would be a given value, stored at `^points['z_1']`.

`record` lists the responses: `f`, `y_1` and `y_2`. None is given a name here, so cottax mints
one for each record: the record of `f`, its values at every point stacked one per point, is
written at `^record['f']`, while `f` itself holds the value at a single point. It is read back
as `out[cx.record('f')]`, and a model downstream reading it names it explicitly,
`@flat.function('f_mean', reads={'fs': cx.record('f')})`. 
Alternatively, the record can be named instead, e.g. `record={'f': 'fs'}`,
the record of `f` then would be written at `fs`, and read back as an ordinary name `out['fs']` by
any model with a parameter called `fs` (e.g. `def f_mean(fs)`).

The DOE owns its factors inside its cycle and reads the responses there; it reads the plans
outside and owns the records outside, as the node shows:

In [ ]:
sweep    = flat.sample(over={'z_1': 'z_1s', 'z_2': 'z_2s'}, record=('f', 'y_1', 'y_2'))
declared = flat.declare(grid, D1, D2, obj, {'sweep': sweep})
declared.graph[flat.node('sweep')]

No architecture preset places a DOE yet, so the analysis is written by hand: $y_2$ is cut for `D1` and
closed by the fixed point $\hat y_2 = y_2$, and that closure is nested in the DOE, so it is solved
at every point. `drive` then picks `JaxDoe` for the DOE and Newton for the closure.

In [ ]:
doe = (declared
       + flat.fixed_point_cut('y_2', 'D1')
       + flat.nest(cx.closure('y_2'), 'sweep')
       + flat.drive(cx.JaxDoe(), cx.Newton(rtol=1e-10, atol=1e-10)))
doe.plan

In the XDSM the DOE's ring encloses the analysis' ring.

In [ ]:
cx.render_html(doe, blocks=True)

## Running it

The run takes $x_1$ and the closure's start; the plan is computed upstream, and each record
comes back stacked, one entry per point.

In [ ]:
out = flat.run(doe, {cx.Inputs: {'x_1': 1.}, Start: {cx.copy('y_2'): 1.}})
f = out[cx.record('f')]
f.shape

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
filled = ax.contourf(out['z_1s'].reshape(N, N), out['z_2s'].reshape(N, N), f.reshape(N, N), levels=12)
fig.colorbar(filled, label='$f$')
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$');

Each point is what one run of the analysis gives there, here at $z_1 = z_2 = 2$:

In [ ]:
analysis = (flat.declare(D1, D2, obj) + flat.fixed_point_cut('y_2', 'D1')
            + flat.drive(cx.Newton(rtol=1e-10, atol=1e-10)))
i = 40
one = flat.run(analysis, {cx.Inputs: {'x_1': 1., 'z_1': out['z_1s'][i], 'z_2': out['z_2s'][i]},
                          Start: {cx.copy('y_2'): 1.}})
one['f'], f[i]

`JaxDoe` maps the points with `jax.lax.map`, one at a time by default; `batch_size` evaluates that
many at once, vmapped. The records are the same:

In [ ]:
at_once = (declared
           + flat.fixed_point_cut('y_2', 'D1')
           + flat.nest(cx.closure('y_2'), 'sweep')
           + flat.drive(cx.JaxDoe(batch_size=N * N), cx.Newton(rtol=1e-10, atol=1e-10)))
jnp.allclose(flat.run(at_once, {cx.Inputs: {'x_1': 1.}, Start: {cx.copy('y_2'): 1.}})[cx.record('f')], f)

## Work in progress

- Care: Only the records, and the plan, are output that is to be used downstream. Anything else on the DOE's cycle, the factors included, hold what the last iteration point left, and nothing refuses a model reading it yet.
- The records are the DOE's own outputs; a rework of driver histories may move them to the
  driver and rename `^record`.
- A DOE is nested in nothing for now, and no architecture (MDA, MDF, ...) places one.
- The plan is written by hand here (`grid`). A `flat.scan` sugar is planned, writing the plan
  model from a preset (a full factorial, a Latin hypercube, ...) over each factor's range.